# Approved Use Case Variable Processing

### Use Case: bmi_statistics_by_wave_race_gender_age_cohort

### 1. Configuration

### 1.1 Execute the Approved Variable Dictionary

Built-in magic command that executes an external Python file as a script within your current session.
Running %run "./Step 1.5.1 approved_variable_dictionary_config" does the following:

Shared variable dictionary for use-case processing notebooks. 

Consumed via **%run** from Step 1.5.2 Approved Variable Processing — this notebook only defines data, it must not execute any table writes.

NOTE: 
* %run executes every cell of the config notebook, and the dictionary cell ends with a print statement — so you'll see its message printed directly under the %run cell in this notebook. From the last run it was:

    * approved_variable_dictionary_a loaded: 5 entries.

In [0]:
%run "./Step 1.5.1 approved_variable_dictionary_config"

### 1.2 Configure Global Variables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, BooleanType
)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

# Use case is a run parameter — select it via the widget (or the job
# parameter when scheduled) instead of editing this cell for each use case.
dbutils.widgets.text("use_case_name", "bmi_statistics_by_wave_race_gender_age_cohort", "Use case name")
dbutils.widgets.text("codebook_version", "1992-2022v1", "Codebook version")

use_case_name = dbutils.widgets.get("use_case_name")
codebook_version = dbutils.widgets.get("codebook_version")

target_table = ("dev_catalog.hrs_agent_metadata.hrs_use_case_approved_variables")

bronze_table = ("dev_catalog.brz_raw_hrs.randhrs1992_2022v1")

# The approved variable dictionary is imported from the shared
# config notebook by the %run cell above
# (approved_variable_dictionary_a). Edit the dictionary there:
#   Step 1.5.1 approved_variable_dictionary_config (same folder)

requested_approval_status = "Approved"

# Do not claim coverage has been approved until it is actually checked.
initial_wave_coverage_status = "Pending"

display(approved_variable_dictionary_a)

### 2. Expand dictionary entries into individual variable records

This determines if the variable is variant (R{wave}BMI) or invariant (RARACEM).

If it is a "variant" variable, then create individual variable records based on the provided range.

In [0]:
# ------------------------------------------------------------
# 2. Expand dictionary entries into individual variable records
#
# ------------------------------------------------------------
expanded_variables = []

for item in approved_variable_dictionary_a:
    root = item["var_name_root"].strip().upper()
    wave_range = str(item["var_wave_range"]).strip().upper()
    stem = item["var_stem"].strip().upper()
    source_page = item.get("var_source_page")
    variable_label = item.get("var_label")

    if not root or not stem:
        raise ValueError(
            f"Variable root and stem are required: {item}"
        )

    if wave_range == "A":
        # "A" means invariant/across-wave variable.
        # Example: R + A + RACEM = RARACEM
        variable_name = f"{root}A{stem}"

        expanded_variables.append({
            "use_case_name": use_case_name,
            "codebook_version": codebook_version,
            "variable_name": variable_name,
            "variable_label": variable_label,
            "definition": None,
            "survey_section": None,
            "measurement_unit": None,
            "source_page": source_page,
            "approval_status": requested_approval_status,
            "wave_number": None,
            "wave_coverage_status": initial_wave_coverage_status
        })

    else:
        # Accept a single wave ("4") or inclusive range ("1-16").
        parts = wave_range.split("-")

        if len(parts) == 1:
            wave_start = wave_end = int(parts[0])
        elif len(parts) == 2:
            wave_start, wave_end = map(int, parts)
        else:
            raise ValueError(
                f"Invalid wave range '{wave_range}' in {item}"
            )

        if wave_start < 1 or wave_end < wave_start:
            raise ValueError(
                f"Invalid wave range '{wave_range}' in {item}"
            )

        for wave_number in range(wave_start, wave_end + 1):
            variable_name = f"{root}{wave_number}{stem}"

            # Apply known metadata defaults for BMI.
            is_bmi = stem == "BMI"

            expanded_variables.append({
                "use_case_name": use_case_name,
                "codebook_version": codebook_version,
                "variable_name": variable_name,
                "variable_label": variable_label,
                "definition": None,
                "survey_section": (
                    "Section B: Health" if is_bmi else None
                ),
                "measurement_unit": (
                    "kg/m²" if is_bmi else None
                ),
                "source_page": source_page,
                "approval_status": requested_approval_status,
                "wave_number": str(wave_number),
                "wave_coverage_status": initial_wave_coverage_status
            })



### 3. Remove duplicate dictionary entries and validate names

In [0]:
# ------------------------------------------------------------
# 3. Remove duplicate dictionary entries and validate names
# ------------------------------------------------------------

# A variable must not be generated more than once for this use case.
seen = set()
deduplicated_variables = []

for record in expanded_variables:
    key = (
        record["use_case_name"],
        record["codebook_version"],
        record["variable_name"]
    )

    if key in seen:
        print(f"Skipping duplicate variable: {record['variable_name']}")
        continue

    seen.add(key)
    deduplicated_variables.append(record)

expanded_variables = deduplicated_variables

if not expanded_variables:
    raise ValueError("No variables were generated from the dictionary.")

print(f"Unique variables generated: {len(expanded_variables)}")


### 4. Validate each variable against the Bronze table

This ensure the variable is a real data variable in the source HRS data file.

In [0]:
# ------------------------------------------------------------
# 4. Validate each variable against the Bronze table
# bronze_table = ("dev_catalog.brz_raw_hrs.randhrs1992_2022v1")
# ------------------------------------------------------------

bronze_columns = {
    column.lower()
    for column in spark.table(bronze_table).columns
}

for record in expanded_variables:
    record["bronze_column_match"] = (
        record["variable_name"].lower() in bronze_columns
    )

matched_count = sum(
    1 for r in expanded_variables if r["bronze_column_match"]
)
unmatched_count = len(expanded_variables) - matched_count

print(f"Exact Bronze matches: {matched_count}")
print(f"No exact Bronze match: {unmatched_count}")

unmatched_variables = [
    r["variable_name"]
    for r in expanded_variables
    if not r["bronze_column_match"]
]

if unmatched_variables:
    print("Variables without a Bronze match:", unmatched_variables)



### 5. Create the staging DataFrame

In [0]:
# ------------------------------------------------------------
# 5. Create the staging DataFrame
# ------------------------------------------------------------

schema = StructType([
    StructField("use_case_name", StringType(), False),
    StructField("codebook_version", StringType(), False),
    StructField("variable_name", StringType(), False),
    StructField("variable_label", StringType(), True),
    StructField("definition", StringType(), True),
    StructField("survey_section", StringType(), True),
    StructField("measurement_unit", StringType(), True),
    StructField("source_page", IntegerType(), True),
    StructField("approval_status", StringType(), False),
    StructField("wave_number", StringType(), True),
    StructField("wave_coverage_status", StringType(), False),
    StructField("bronze_column_match", BooleanType(), False)
])

variables_df = spark.createDataFrame(
    expanded_variables,
    schema=schema
)

display(
    variables_df.orderBy("variable_name")
)


### 6. Confirm target table columns and upsert to Delta Table

Update the Variable Review table with the updated column values.

In [0]:

# ------------------------------------------------------------
# 6. Confirm target table columns and merge
# 
# target_table = ("dev_catalog.hrs_agent_metadata.hrs_use_case_approved_variables")
# ------------------------------------------------------------

target_columns = {
    column.lower()
    for column in spark.table(target_table).columns
}

required_columns = {
    "use_case_name",
    "codebook_version",
    "variable_name",
    "variable_label",
    "definition",
    "survey_section",
    "measurement_unit",
    "source_page",
    "approval_status",
    "wave_number",
    "wave_coverage_status",
    "bronze_column_match",
    "created_at",
    "updated_at"
}

missing_columns = required_columns - target_columns

if missing_columns:
    raise ValueError(
        "Target table is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

# Protect against an ambiguous Delta MERGE if the target already
# contains multiple records for a logical variable key.
duplicate_targets = (
    spark.table(target_table)
    .filter(F.col("use_case_name") == use_case_name)
    .filter(F.col("codebook_version") == codebook_version)
    .groupBy("use_case_name", "codebook_version", "variable_name")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_targets.limit(1).count() > 0:
    display(duplicate_targets)
    raise ValueError(
        "Duplicate target keys exist. Resolve them before MERGE."
    )

variables_df.createOrReplaceTempView("approved_variable_updates")

spark.sql(f"""
    MERGE INTO {target_table} AS target
    USING approved_variable_updates AS source
      ON target.use_case_name = source.use_case_name
     AND target.codebook_version = source.codebook_version
     AND target.variable_name = source.variable_name

    WHEN MATCHED THEN UPDATE SET
        target.variable_label = source.variable_label,
        target.definition = source.definition,
        target.survey_section = source.survey_section,
        target.measurement_unit = source.measurement_unit,
        target.source_page = source.source_page,
        target.bronze_column_match = source.bronze_column_match,
        target.wave_number = source.wave_number,

        target.approval_status =
            CASE
                WHEN target.approval_status = 'Approved'
                    THEN target.approval_status
                ELSE source.approval_status
            END,

        target.wave_coverage_status =
            CASE
                WHEN target.wave_coverage_status = 'Approved'
                    THEN target.wave_coverage_status
                ELSE source.wave_coverage_status
            END,

        target.updated_at = current_timestamp()

    WHEN NOT MATCHED THEN INSERT (
        use_case_name,
        codebook_version,
        variable_name,
        variable_label,
        definition,
        survey_section,
        measurement_unit,
        source_page,
        bronze_column_match,
        approval_status,
        wave_number,
        wave_coverage_status,
        created_at,
        updated_at
    )
    VALUES (
        source.use_case_name,
        source.codebook_version,
        source.variable_name,
        source.variable_label,
        source.definition,
        source.survey_section,
        source.measurement_unit,
        source.source_page,
        source.bronze_column_match,
        source.approval_status,
        source.wave_number,
        source.wave_coverage_status,
        current_timestamp(),
        current_timestamp()
    )
""")


### 7. Verify persisted records

In [0]:
# ------------------------------------------------------------
# 7. Verify persisted records
# ------------------------------------------------------------

persisted_df = (
    spark.table(target_table)
    .filter(F.col("use_case_name") == use_case_name)
    .filter(F.col("codebook_version") == codebook_version)
)

display(persisted_df.orderBy("variable_name"))

print("Approved-variable dictionary processing completed.")